In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-02-17 13:48:43.407228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771364923.608702 1903525 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771364923.662901 1903525 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771364924.090356 1903525 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771364924.090381 1903525 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771364924.090384 1903525 computation_placer.cc:177] computation placer alr

In [2]:
# minorized reference
with h5py.File('/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5', 'r') as file:
    #print("Keys: %s" % file.keys())
    ref_d = file['density'][:]
#print(ref_d)
print(ref_d.shape)




(64, 64, 64)


In [3]:
import os

# Get the current working directory
current_directory = os.getcwd()

# Print the current working directory
print(current_directory)


/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Data_Gen


In [4]:
####################################################################################################################################################
def stochastic_density(d,N):
    # poisson model
    #  accurate and fast for all values of N
    # N  = number of MC samples
    assert isinstance(d,np.ndarray)
    assert isinstance(N,(int,float,np.int64,np.float64))
    assert N>0
    ds = np.random.poisson(N*d)/N
    ds*= d.sum()/ds.sum()
    return ds
#end def stochastic_density

####################################################################################################################################################

In [6]:
import numpy as np

# ==========================================
# 1. SETUP & PARAMETERS
# ==========================================
num_train = 1000
num_val = 200
samp = [165150720,41287680,20643840] #samples to match
# samples_list = [10000000] # 10 M already done
samples_list = [1000000 , 10000000, 100000000, 1000000000, 10000000000]

# --- PLACEHOLDER: Ensure your 'ref_d' and function are loaded here ---
# ref_d = ... (Your (116, 116, 72) numpy array)
# def stochastic_density(density, samples): ...
# =====================================================================

def transform_pair(x_vol, y_vol):
    """
    Applies identical random transformations to input (x) and target (y).
    This forces the CAE to learn local features rather than memorizing 
    the single geometry's coordinates.
    """
    # 1. Random Flips (Mirroring)
    # Flip x-axis?
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=0)
        y_vol = np.flip(y_vol, axis=0)
    # Flip y-axis?
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=1)
        y_vol = np.flip(y_vol, axis=1)
    # Flip z-axis?
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=2)
        y_vol = np.flip(y_vol, axis=2)

    # 2. Random Translation (Rolling)
    # CRITICAL: Only use this if your box has Periodic Boundary Conditions (PBC).
    # If your molecule is isolated in the center and edges are zero, 
    # you might want to comment this section out to avoid splitting the molecule.
    shift_x = np.random.randint(0, x_vol.shape[0])
    shift_y = np.random.randint(0, x_vol.shape[1])
    shift_z = np.random.randint(0, x_vol.shape[2])
    
    x_vol = np.roll(x_vol, shift=(shift_x, shift_y, shift_z), axis=(0, 1, 2))
    y_vol = np.roll(y_vol, shift=(shift_x, shift_y, shift_z), axis=(0, 1, 2))

    # 3. 90-Degree Rotations (XY Plane)
    # Since z (72) is different from x,y (116), we only rotate the square plane.
    k = np.random.randint(0, 4) # 0, 1, 2, or 3 rotations
    x_vol = np.rot90(x_vol, k=k, axes=(0, 1))
    y_vol = np.rot90(y_vol, k=k, axes=(0, 1))

    return x_vol, y_vol

def generate_dataset(ref_d, base_samples, num_train, num_val):
    """
    Generates x (noisy) and y (cleaner) pairs using the single reference geometry,
    but augments them so they look spatially distinct.
    """
    total_mc_samples = 0
    
    # Define noise levels (samples)
    input_sample_opts = [base_samples * (j + 1) for j in range(3)] 

    def create_batch(n_items):
        nonlocal total_mc_samples
        # Use float32 to reduce memory footprint (approx 50% savings)
        x_data = np.zeros((n_items, 64, 64, 64), dtype=np.float32)
        y_data = np.zeros((n_items, 64, 64, 64), dtype=np.float32)
        
        for i in range(n_items):
            # A. Select noise levels
            n_samples_in = np.random.choice(input_sample_opts)
            n_samples_out = n_samples_in * 100 # Target is 100x cleaner
            
            # Track total computational cost
            total_mc_samples += (n_samples_in + n_samples_out)
            
            # B. Generate raw noisy data from the SINGLE geometry
            # Note: We generate fresh noise every time so the noise pattern is unique
            raw_x = stochastic_density(ref_d, n_samples_in)
            raw_y = stochastic_density(ref_d, n_samples_out)
            
            # C. Augment the pair together
            # This makes the single geometry look like many different ones
            aug_x, aug_y = transform_pair(raw_x, raw_y)
            
            x_data[i] = aug_x
            y_data[i] = aug_y
            
        return x_data, y_data

    print(f"  - Generating {num_train} Training pairs...")
    x_train, y_train = create_batch(num_train)
    
    print(f"  - Generating {num_val} Validation pairs...")
    x_val, y_val = create_batch(num_val)

    return {
        "x_train": x_train,
        "x_val": x_val,
        "y_train": y_train,
        "y_val": y_val,
        "total_samples": total_mc_samples
    }

# ==========================================
# 2. MAIN EXECUTION LOOP
# ==========================================
for samples in samples_list:
    print(f"Generating data for base sample scale: {samples}")
    
    # Run the generator
    data_dict = generate_dataset(ref_d, samples, num_train, num_val)

    # Construct filename
    file_name = f"{data_dict['total_samples']}_sample_training_data_Aug.npz"
    save_path = "DFT_W_N/DFT_dat_lev_" + file_name
    
    # Save compressed (.npz)
    np.savez(save_path,
             x_train=data_dict["x_train"],
             x_val=data_dict["x_val"],
             y_train=data_dict["y_train"],
             y_val=data_dict["y_val"],
             total_samples=data_dict['total_samples'])

    print(f"Saved dataset to {save_path}")
    print("-" * 30)

Generating data for base sample scale: 1000000
  - Generating 1000 Training pairs...
  - Generating 200 Validation pairs...
Saved dataset to DFT_W_N/DFT_dat_lev_249369000000_sample_training_data_Aug.npz
------------------------------
Generating data for base sample scale: 10000000
  - Generating 1000 Training pairs...
  - Generating 200 Validation pairs...
Saved dataset to DFT_W_N/DFT_dat_lev_2414910000000_sample_training_data_Aug.npz
------------------------------
Generating data for base sample scale: 100000000
  - Generating 1000 Training pairs...
  - Generating 200 Validation pairs...
Saved dataset to DFT_W_N/DFT_dat_lev_24048100000000_sample_training_data_Aug.npz
------------------------------
Generating data for base sample scale: 1000000000
  - Generating 1000 Training pairs...
  - Generating 200 Validation pairs...
Saved dataset to DFT_W_N/DFT_dat_lev_242097000000000_sample_training_data_Aug.npz
------------------------------
Generating data for base sample scale: 10000000000
 